
`create` a python spark random generator to create cdc kafka
with all kafka fields 
topic=clinet
in one topic 2 tables clinet and clinet_address under single topic `clinet`
data will be in raw json fomat 
in this location: corpadmin.imptables


In [0]:
import json
import random
from datetime import datetime, timedelta
from pyspark.sql import types as T

random.seed()

TARGET = "corpadmin.imptables.sqlkafkastream"   # raw Kafka CDC landing table
TOPIC = "clinet"                               # single topic carrying both tables' CDC
N_PARTITIONS = 3
CONSUMER_GROUP = "cdc-sqlserver-sim"

# ---- landing table: one row = one raw Kafka record (all consumer fields) ----
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TARGET} (
  kafka_key           STRING COMMENT 'Kafka record key = client id',
  kafka_value         STRING COMMENT 'Raw JSON CDC payload exactly as consumed from the topic',
  kafka_topic         STRING COMMENT 'Kafka topic (always clinet)',
  kafka_partition     INT COMMENT 'Kafka partition',
  kafka_offset        BIGINT COMMENT 'Kafka offset within the partition',
  kafka_timestamp     TIMESTAMP COMMENT 'Kafka record timestamp (CreateTime)',
  kafka_timestamp_type INT COMMENT '0 = CreateTime, 1 = LogAppendTime',
  kafka_leader_epoch  INT COMMENT 'Leader epoch of the broker that appended the record',
  kafka_headers       ARRAY<STRUCT<key: STRING, value: STRING>> COMMENT 'Kafka headers attached to the record',
  consumer_group      STRING COMMENT 'Kafka consumer group that read the record',
  consumed_at         TIMESTAMP COMMENT 'When the consumer processed the record'
)
CLUSTER BY (kafka_topic)  -- liquid clustering by topic: file-level pruning for per-topic ETL reads when more topics land
COMMENT 'Raw Kafka CDC capture from topic clinet: SQL Server CDC for source tables client (id, clinet_name, type) and client_address (clinet_id, address, city, country), payload kept as raw JSON'
""")

# ---- shared CDC helpers: live state from the event store, event emission, append ----
def load_state():
    """Rebuild the current live client/address state and counters from the bronze event store,
    so the CREATE / UPDATE / DELETE cells below can run independently, in any order."""
    clients, addresses, seq_max, max_cid = {}, {}, 0, 0
    for r in spark.sql(f"""
        SELECT kafka_value FROM {TARGET}
        WHERE kafka_topic = '{TOPIC}'
        ORDER BY CAST(get_json_object(kafka_value, '$.seqval') AS BIGINT)
    """).collect():
        v = json.loads(r["kafka_value"])
        seq_max = max(seq_max, v.get("seqval") or 0)
        key = (v.get("after") or v.get("before") or {}).get("id") or (v.get("after") or v.get("before") or {}).get("clinet_id")
        if not key:
            continue
        max_cid = max(max_cid, int(key[1:]))
        if v["table"] == "client":
            if v["op"] == "d":
                clients.pop(key, None)
                addresses.pop(key, None)
            else:
                clients[key] = {"clinet_name": v["after"]["clinet_name"], "type": v["after"]["type"]}
        else:
            if v["op"] == "d":
                addresses.pop(key, None)
            else:
                addresses[key] = {"address": v["after"]["address"], "city": v["after"]["city"], "country": v["after"]["country"]}
    part_offsets = {p: -1 for p in range(N_PARTITIONS)}
    for r in spark.sql(
        f"SELECT kafka_partition AS p, MAX(kafka_offset) AS m FROM {TARGET} GROUP BY kafka_partition"
    ).collect():
        part_offsets[r["p"]] = r["m"]
    return {"clients": clients, "addresses": addresses, "seq": seq_max,
            "next_cid": max_cid + 1, "offsets": part_offsets,
            "ts": datetime.now() - timedelta(seconds=60)}

def emit(st, tbl, op, cid, before=None, after=None):
    """Build one CDC event: raw JSON payload plus all Kafka consumer fields."""
    st["seq"] += 1
    st["ts"] += timedelta(seconds=random.randint(2, 30))
    payload = {
        "table": tbl, "op": op,
        "lsn": f"0x{random.randint(0x100000, 0x200000):08X}:{random.randint(1, 9999):04X}:0001",
        "seqval": st["seq"], "ts": st["ts"].isoformat(),
        "before": before, "after": after,
    }
    part = client_partition(cid)
    st["offsets"][part] = st["offsets"].get(part, -1) + 1
    return {
        "kafka_key": cid, "kafka_value": json.dumps(payload), "kafka_topic": TOPIC,
        "kafka_partition": part, "kafka_offset": st["offsets"][part],
        "kafka_timestamp": st["ts"], "kafka_timestamp_type": 0,
        "kafka_leader_epoch": random.randint(0, 5), "kafka_headers": make_headers(tbl),
        "consumer_group": CONSUMER_GROUP, "consumed_at": datetime.now(),
    }

def append_events(events):
    """Append generated CDC events to the landing table and show a sample."""
    header_t = T.StructType([T.StructField("key", T.StringType()), T.StructField("value", T.StringType())])
    schema = T.StructType([
        T.StructField("kafka_key", T.StringType()),
        T.StructField("kafka_value", T.StringType()),
        T.StructField("kafka_topic", T.StringType()),
        T.StructField("kafka_partition", T.IntegerType()),
        T.StructField("kafka_offset", T.LongType()),
        T.StructField("kafka_timestamp", T.TimestampType()),
        T.StructField("kafka_timestamp_type", T.IntegerType()),
        T.StructField("kafka_leader_epoch", T.IntegerType()),
        T.StructField("kafka_headers", T.ArrayType(header_t)),
        T.StructField("consumer_group", T.StringType()),
        T.StructField("consumed_at", T.TimestampType()),
    ])
    sdf = spark.createDataFrame(events, schema)
    sdf.write.mode("append").saveAsTable(TARGET)
    from collections import Counter
    print(f"Appended {len(events)} CDC events to {TARGET} (topic {TOPIC!r}, consumer group {CONSUMER_GROUP!r})")
    print("Breakdown:", dict(Counter((json.loads(e['kafka_value'])['table'], json.loads(e['kafka_value'])['op']) for e in events)))
    display(sdf.orderBy(sdf.kafka_offset.desc()).limit(10))

def live_keys(tbl):
    """kafka_keys for source table `tbl` that are PRESENT in the table and currently live:
    they have at least one event and their latest event is not a delete.
    UPDATE / DELETE cells only operate on keys from this set."""
    return {r["kafka_key"] for r in spark.sql(f"""
        SELECT kafka_key FROM (
            SELECT kafka_key,
                   get_json_object(kafka_value, '$.op') AS op,
                   ROW_NUMBER() OVER (
                       PARTITION BY kafka_key
                       ORDER BY CAST(get_json_object(kafka_value, '$.seqval') AS BIGINT) DESC
                   ) AS rn
            FROM {TARGET}
            WHERE kafka_topic = '{TOPIC}'
              AND get_json_object(kafka_value, '$.table') = '{tbl}'
        ) WHERE rn = 1 AND op <> 'd'
    """).collect()}

# ---- reference data for realistic random values ----
FIRST = ["Acme", "Globex", "Initech", "Hooli", "Stark", "Wayne"]
SUFFIX = ["Corp", "Ltd", "Group", "Holdings", "Industries"]
TYPES = ["individual", "corporate", "government"]
CITIES = [("Mumbai", "India"), ("London", "UK"), ("New York", "USA"), ("Tokyo", "Japan")]
STREETS = ["MG Road", "Baker Street", "5th Avenue", "Elm St", "Park Lane"]

def client_partition(key: str) -> int:
    """Kafka-style partitioning: the numeric part of the client id maps to a stable partition."""
    return int(key[1:]) % N_PARTITIONS

def make_headers(tbl: str):
    hs = [("db", "sqlserver"), ("schema", "dbo"), ("table", tbl), ("topic", TOPIC)]
    if random.random() < 0.5:
        hs.append(("cdc", "true"))
    return [{"key": k, "value": v} for k, v in hs]

# Event generation is split into dedicated cells below:
#   CREATE events cell | UPDATE events cell | DELETE events cell
# Run this cell first (it defines the helpers and ensures the table exists),
# then run any op cell, in any order, any number of times.

In [0]:
# CREATE events: 4-8 new clients + 1-2 new addresses = 5-10 events per run
st = load_state()
events = []

# new client records (source inserts)
for _ in range(random.randint(4, 8)):
    cid = f"C{st['next_cid']:05d}"
    st["next_cid"] += 1
    new = {"clinet_name": f"{random.choice(FIRST)} {random.choice(SUFFIX)}", "type": random.choice(TYPES)}
    st["clients"][cid] = new
    events.append(emit(st, "client", "c", cid, None, {"id": cid, **new}))

# new addresses for existing clients that have none
no_addr = [c for c in st["clients"] if c not in st["addresses"]]
for cid in random.sample(no_addr, k=min(random.randint(1, 2), len(no_addr))):
    city, country = random.choice(CITIES)
    addr = {"address": f"{random.randint(1, 300)} {random.choice(STREETS)}", "city": city, "country": country}
    st["addresses"][cid] = addr
    events.append(emit(st, "client_address", "c", cid, None, {"clinet_id": cid, **addr}))

append_events(events)

In [0]:
# UPDATE events: 3-6 client updates + 2-4 address updates = 5-10 events per run
st = load_state()
events = []

# condition: update only kafka_keys that are present in the table and live (latest event is not a delete)
live_clients = live_keys("client")
live_addresses = live_keys("client_address")

# client updates: at least one field always changes, so every event creates a new SCD2 version
for _ in range(random.randint(3, 6)):
    candidates = [c for c in st["clients"] if c in live_clients]
    if not candidates:
        print("No live client kafka_keys present in the table - run the CREATE cell first")
        break
    cid = random.choice(candidates)
    before = {"id": cid, **st["clients"][cid]}
    after = dict(st["clients"][cid])
    if random.random() < 0.7:
        after["clinet_name"] = f"{random.choice(FIRST)} {random.choice(SUFFIX)}"
    if after == st["clients"][cid]:
        after["type"] = random.choice([t for t in TYPES if t != st["clients"][cid]["type"]])
    elif random.random() < 0.4:
        after["type"] = random.choice(TYPES)
    st["clients"][cid] = after
    events.append(emit(st, "client", "u", cid, before, {"id": cid, **after}))

# address updates
for _ in range(random.randint(2, 4)):
    candidates = [c for c in st["addresses"] if c in live_addresses]
    if not candidates:
        print("No live address kafka_keys present in the table - run the CREATE cell first")
        break
    cid = random.choice(candidates)
    before = {"clinet_id": cid, **st["addresses"][cid]}
    after = dict(st["addresses"][cid])
    if random.random() < 0.7:
        after["address"] = f"{random.randint(1, 300)} {random.choice(STREETS)}"
    if after == st["addresses"][cid]:
        after["city"], after["country"] = random.choice(
            [c for c in CITIES if c != (st["addresses"][cid]["city"], st["addresses"][cid]["country"])])
    elif random.random() < 0.4:
        after["city"], after["country"] = random.choice(CITIES)
    st["addresses"][cid] = after
    events.append(emit(st, "client_address", "u", cid, before, {"clinet_id": cid, **after}))

append_events(events)

In [0]:
# DELETE events: 3-5 client deletes + 1 address-only delete, capped at 10 events per run
st = load_state()
events = []

# condition: delete only kafka_keys that are present in the table and live (latest event is not a delete)
live_clients = live_keys("client")
live_addresses = live_keys("client_address")
client_candidates = [c for c in st["clients"] if c in live_clients]

# client deletes (address rows for the same client are removed with it, as in the source system)
for cid in random.sample(sorted(client_candidates), k=min(random.randint(3, 5), len(client_candidates))):
    before = {"id": cid, **st["clients"][cid]}
    st["clients"].pop(cid)
    st["addresses"].pop(cid, None)
    events.append(emit(st, "client", "d", cid, before, None))

# one address-only delete (kafka_key must be present and live in the table)
addr_candidates = [c for c in st["addresses"] if c in live_addresses]
if addr_candidates:
    cid = random.choice(addr_candidates)
    before = {"clinet_id": cid, **st["addresses"][cid]}
    st["addresses"].pop(cid)
    events.append(emit(st, "client_address", "d", cid, before, None))

if events:
    append_events(events)
else:
    print("Nothing live to delete - run the CREATE cell first")

In [0]:
%sql
select * from
      corpadmin.imptables.sqlkafkastream  order by kafka_key
 
  